# CIC-DDoS2019 LightGBM CPU baseline

Smoke: 2,000 rows per source file and at most 10 new iterations in this session; target remains exactly 100. Resume/checkpoint state is synchronized with S3.


In [ ]:
from pathlib import Path
import base64
import json
import os
import subprocess
import sys
import zlib

PROJECT_NAME = "Luan-Van-LightGBM-Parquet-Github-v2"
PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
SOURCE_DIR = PROJECT_DIR / "source"
PREPARED_DIR = PROJECT_DIR / "prepared"
RUNS_DIR = PROJECT_DIR / "runs"
for directory in (SOURCE_DIR, PREPARED_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

encoded_files = json.loads("{\"checkpoint.py\": \"eNrtPGtv3EaS3/UruASEkN4RJdvJYjG5OVzs2I6xdmzYzt0tdAJBDXskrjjkLB+WFa3++1VVP9gvznAke28/nIFEQ3Z3dXW9q7qbYRg+b7L28qjNVix4U1xcdq+evQ2e1XXbsSZYXrLl1aYuqq4NsioPPrOmWBUsD7KuXhfL4OPToL2plkkYhgcHq6ZeB2m66ru+YWkaFOtN3XQwrqq7rCvqqj04EO+W7Wf58xJmL4tz+fi3tq7k77K+uCiqC/lYt/JXe9l3RSmfumLN5O++L3KOSJ51DFskGvJ5Rv1/ryvG+22yDueX3d7DI2/objYwuXz/U3UzC95mG3w3Cz6yv/esWrKDg4M37169evEhWEhskwvWvYGfrInStMrWQIkYuuVsFfTdMq3q6ygOjv49aLtmfhDAv4YBvSqFX4I9JIoJDImToq1XdbPOukhCai+zJz/8KV0VJYtwAXPCewb86qur9PymY+08AK4BWo9PnnwfPKI/5rx5ccFa7CE4kHCgMAe2XhfdJdEmqTesisLmPIyDrIXOVV4yDoH6XQIOwXlZL6+C+UI0Jw3L8khDJh4GDFMn/QYXHdHg2KAFb79kX/gvte5lVtVVsczKFJGGpd+UdZbPkTvm4oA5dQ5yuiCBSvJ+vWll9xm0tiiiWbssisXLrGxBKlrgcnrFbtrFp6bHZ7bJGhDzpl1E4SycBeE8jOOEA47Cvlsd/Tk0sLboKFCIPcvg2pMiaimiZixkFuTQt6hIZThjaW2/gjwIzg3tCSDJqi5ZX+VFE/EHuQL2pWi7tL6iR45px1Cas+YGCKNDQW6nbb9aFV8i/T1/FfwxCJNuvQk10VCQhHxchzNOdFCBhaSOT2AUPwZ28C5+thRVDktaPLEZFCuAQuSumwJkKfyfKnSaVmUPwjK8rttkhWYrku0gwlUdxbwHtDZsU2ZLFqlFGjyxuHgJVK6bG85I8TBXFuJU2IxTEM0ZsvfsbEY0SA29bT9rzxa7HWkpYRI5U6xB4/hLWA8QDQVClwv10isU4BXKHM0dGB7ED9d7BqBOz3hz3YCaLOsmB44GkkqKI9gMrMU23su0F8WKWsGPYA9tKqOXiUYCdGdVHsHAPcV2FlTsuiwqtghHbB6KWsPplPxcLLv/oheRlOMBicXwM7aGc4G9BEMJI/2NTX3dKj5/C6GWLJUS/TkrCzTJUqYnivOyb1C4UkRZWC3wPZYUq8YWpQLaI87p01C1hGexX1C4DLEvG7bsyKiTBjRZdcGix2gjusjBIQYBfSxWD/KjTf+HhQI18LTJipYF/5mVPXvRNHUTGaK1CgUmCSpbsO7BcS7rqssAR/xbVH3dt0FfFUAmfa7HSXLroHb3YxBa4OvzljWfYW0oEIvbAcLp/IezuwAmKo23Rz/Mz+4GIMjAZZm1LURkHwFPQXMIyyBCy/Jsg/LatxjPDHreb7j55UHdkiADM+sNSMYQ7PH3PMLjzmcFQV5RFV2aRi0rVzikWhUX88CRDdSt7LxkeVoDsKbI2Tw4r+sy+AcJBjAS/1iCQjaMIIIzQGbjiIi/OQ0FQBAV1RmRSMR7VMphLPDdRiAoWj43A/fCQdtdLMgNiH7hwYSTL5XNDkbr7As0dk3BUOLhSYmqAKB1gMHOrB34lPOsZWkL6lDlCGQFMw7j3S4ODuf98ophjAf6z6rPRQOchPg0Ag4pMLxPCs0wHMKcME6gudhEFqxNw9Dsb4XF+/hgib/hseaeG3YBorUdIu+zFTvgMnoG3tN0CVNmWGVleZ4tr9IJUwnWCKDiB1gslCiHWm1xUYGJgWh22Q06Ycq+GqT6k0F2MQZFTt9/ePHx9atfX/ycPn/368vXr9L3P336JRwlignSJAw5Q4w1IrNXzP3ipEjOv14ZcqNyCF8UG0MG/UyVeCI/TDBSLnWRtkYrgfSNbp+KdgDgkz1BI11L0Obpz39YuLi6q3ccR/hekUIOJxMaCLB5sVqxpg0oyQSuPvvt+V9efBpBTaxRoSaeTdT4y4egJsDaqIHAvXz93xZqpmFxKHTgEw3OJwdlU2XSZVmAn6QkaERLJFmktUe6RKj8OlKgj+oVn8dKPyEaa7MLBB6KAsYlqFnxO6eFMOjtwBuaZiCHl1HSDYxx4UNfYVLP+SAQMOnKCwnJddZUoHlReNj+GEBmnJVGFaZhaww6pBucBV5YlkukXIq77//YNKDhTXejnDmnOikQeWKg/twhuOCN9J7mKjs9lFcjed3kvO7qp0Yj+7Jkmy54Te1EDzQu8HYS6UICiHgov3x9yarAw0joI8kUc5GGSVwyyaUtOK6JoAdYEKCu8AkYwS8046/7MMr9dUhDnAS0/hvEmZi1ilCpYSUg9xniILBJZsmCQ+PN5Fz4z6T0ek6adRXealJ+d3wrB92FtgWhcEc2Dxg2faVjB09FTrhNQHVYuL5OwImD0dCRa/ju+Lv4DpYxBJIUw4jZUS5F+kAILIVjHso7hmhCuAtRCwrFPHhBMoVM99sNzCqyDkNfyiBV5uAEapgzTBBusXiJYRT7JHxAaky+hyUEC0c4hfpJtP/NQdaFx8laZpjDj0WRj4InwaNHQSThHsGKvYBsewQadtgqdA7z48M8WEFeAhoYHbbxjwFNRiXTKjhMHq/aUOPpTI50iT7DUiuLYP1xIkumM74OP2ZUJm1LxjaR1QsyIIaF44GuYATQGRjSwM3K0MmQR0jg2q4dMYeOMAgrJ4cd7GnkfAZOwjJsHArwEGIBKzqQ4Radtc+2CfFUWA3rG6K9vL6uKEbjygeqOyi6kchRIecfltkXhLZCPx+xP2P4IeVRdeUxrcQBJeX2LqaXql4jE3fIWtsuq5YsIlCEUGyH+bTe2xAAhPNA9Asxl5aPdzZx6K2XMpus6VKe3QnitHXfLJks1q2KKitTg2BaUjtgBinzbwQF4gcQlHXfoTNCNoK9XILbRl3ByYIO3FZ/cRlkIhA7Kot1gZWO9+8+fqLUe3+6UyEluxaRFmbiILyomj5GaEuWvID5121kEZr6GXH3MEWcNONxtqIZhq7gIjRA4Cc4fRNU/bvQNWuiwsJ222VKawS07fsWpskEV12DNUCsIlfr7X+iwtYydhWdxKPdjNBAGZY4wami0VF89wJWdxr2TRmezbb2zLMuW2CCGYkxVHDEQsD2cVgdbBe3If4FFYk0Dsh6fHy3HQSarLrvFo//fHIy3lM4e5c27aauWmUYRCAQalooGQLyqH7HB2OAEjKlKUh9Ctai69soHvPaWPO+n7/eHewro36ww516R2r5GZh4Mg1cKYWjlQkBtfBMAD0y+OU5/Bf6ucBZ62/zON4Dl4MjdORZhfWSbBDPNJAPxXLNuss6H+wsGCrgUl/lOK8WuSk7qoeYSkjASkH80OAYMOyygZsq1+0JkSDT9kE8vQUIqC2aqwl/+fTp/UeSlud1zsBQLBbB9yffYwaJtu3AK2g0kny1DooAUClmKMqitb0NASA2/Fp/7JeXf2E3/KF7iTQI7wYPVJ9TDE1bM63PK5vEEVUvPdPzOkWTS1pW4PoOczx3GFI/PYGD4ak1lGjgfJrUbLEFDgDNOJBnn3ni6fV5ns1dk4v9pYWksWezwX49ObGMpbMqxXtuWlLc2BWyco917jJVPjPlUNSwmriklMsPSJdBBZ6SJlqP6BmVSRZayWQWgFwukJ+7MZlkLFXZwND2eB9ikRFVyiGK7XwrdJ+o7KEK0xa/o2Bi3V74SOQZ1bNTbBsIRlWatl+jGGvHM/io+MEaaEVsZrCGrxQB9lFKisMAcl92HlU3ImHuRoKRaWTNWYM3ui6L4NoYp9tQqluB0dRSIC4JR8I9qjSTwt5bheNd+BAfjlqxpWDn8eqj/bYYBZHtLRyr9U3D3b5L1U7gVw955RoSmEaaXWNGtL8Uusotcz2Y3C9kHMVFjyWNuceDVaOfv9vDIlAFnDzYlJVYTm/nMqTxVyywXZ/JhHEn6F+suYApS8YUHoynNZCfe2hl7FR1gOHRG1ZddJdg0o4exzEmimhgt+nn63fOvsawvY0VYTTe+ja2iKPD2JNjbm7SfURMYwyNJZszlSmoFwc7sj8NKjBKEGxBjZxs1EG8535g4OefTk784juFyQYtpvCY29v7ivRelHPEWVFoX1EekN5DjLVB30yEaY49xXf7YU532Res25NVMGI/TiFdBKOGsWdUvmPZWhxEM+z+dK4BwElyCcEAP5pKx8bkSDxHky45x8QpVST1Qj8q6+cZSAHBG3eUo0dbLTDuEVfa1BUx5H0l5uMvPx0B2ycKzb9SPcQsfXirHhAmFqts6a18yCqHp6KxRzBGdCw9YZA3+tLDtyRnJQOGC4HnT+Ou1oFlk3w+aUvmed2XOQXZfEItPh78IM/6SBeIUp4AfnDOWKGlAu0QQuMhz6NbPFif4P++p3T6ixZZO8TxUmu3rRE5HpUkt9sYPbPVRo27VDohwhOxWWAkvsbSxyPDF1+6JvupucCaqaokzYPbkNtZ+ClV926kbHqP8GqShbajwnun/wYl4gnIeiO7U+kKhSc8U14Qq2pWFEiu01eYE1Td2yauwk96/OcxgzxBNNZ65zGNu+lOQZKsuUyWVm3UuLR6ODTaFzmndHW823OY9yMpAAgwhx/OTU0IsTY5NxViyxaAZNvPYPiXuMm/CJ+/e//X8AHSv2cYqQeP95b6LeWMCUHgdmnXgsVvIOkvpdMflXKtDPIv7vz/b1y+192POy8hVcK339+a7vb1O/28keWizyaCWHOq4qms2IuCpLGz4LltNFPHDsRpbXGA7GtuQqhR5q4H4j13q+/bj9a9hFXJDRWpHB/kwQmgjwiAcEoqRcO6n86Pj281nt0d33rUxIv8/S9fPfACluSihqR7HGV6cfl+Wzy7K8o7mTZSYrXquxKF4LcPb7g1G7FjWwP6feuGoUuDKc5d32oy096HJ7z71GH8V5wmlISNRFnNeP88eVqu7NzeG0mXJ9zIm+rdjG0zwy6OxBCu7VRmHgMuyiwUUrG1f7Yn4j6kvT4KSDtwWZhODyeGPn1VFhUeUxwOfmJshAf1fTvN3hsLW62+ceh6l03Xjwr8UzamxakpcQTKc1jqntvQioj77EW7BmLqTvSeG8iqO7E5Pth19gardFs3kYcOE/eQBSLqPkgbyblPw2d1fhOe8ZviMQRVxrXq4T7bc3X2/W1WZRdMXl7Xb6EZ1wgGAgJJcUOnqeuORBvkmMIb1WENU5YUMvJQyNzKpUnT8bttw6xPJ/aachmOD/BcieOaNSwJ+tP9He2VfQFNLQ/vn6kHs5OzVAxM8DyZ02BBb59CT3HbMFIkmPnWaY3MKF1MCTZdo3WWdhyEolPTVyS+oW652rrkTWmRq3P1pF8sl5y27xcaZ9dFfMIHHIxvYLJcE+V13SmVaZ8mpgHFN/opeAf/OFYmDrpKq0mn8m07KKbCSyf8p8gUSdVDcaoIT5FmDcgaHieCWSrxc5P1Ld0LCRHBGzIS0LVuuvDOu1Q6xEnTnIacpPqVK4mzybMRn8O3oz3d979TxtO/XVfJADvq+G0IZBGJZvLRyLqMMWjaXXprfNWDLqKt8DH67vCvh+vD/Ojwl8O3eCfCvJGB6YRzI4PE+L1xk08/YWrqDx83QG3BdOM74S881z14i7x6MeVGN+2YKyW+NWgnyTQXk5huTbJqLma1Wp3Zw/nIHW9rIN91wWNCMEJ9bWXoNBwCd7/AwRcy88mvqw6DCntOZUGje1zJAjkLJloNTxaPR2XWjFjJRsTEOkfJRUpdChESxgdpxhkByuuncgyYYu36WYjPHYQfqE0cA2Gexyk0XF6Ed+4hR7Kl4haSkBVjymPfdHHsp7snWr8X+JlGjKEcYuXyg8lTfe9zexYH77R0D1gg3XMho5R0Xzpc2xbeWp23rn1PnNZ422fZHutfUrCwEV0IE6PbbjxSqoLeqCSgjQRcwcrdmZYy9YqZ0sttPc88Nq+spRE8T2ToMabcc+xWQItJw5dBxJdoXAijhDX0Vju1OKDCK9q4iNOQvxUl6zPfVSa1Of2zoAfov4mu2qPOa8YzsXXWLS8Drx6OoaevVEdQvt8XRePrHnshqASBNGCbTe7AKbBTM7We8W/lmC/P3Iz7n2zBKZN39WB3Fv8wc7jbFBL8aIpg7/shAzF46+x7Gpqxr+nwr4AIiXVjnTM3gaa+M4mjFtVlECwMzB1LhPU4T+vStni12Wk459/+498is23L3CuwQzf8Ztq6pa+zWWBXLKPvAraA7zrz9eiyBisMZsw5M4RbBqhDkuzXKvpQiqb9MLbhXw6iYHzDmhUkrD1GYnphZJqWaZkx7z2iaYNd50Lq9pbSOwJ6n40FR47EoXXBzcRpj+IpkuqG3J5DI2Tbg4VNGMdT0efCbDlLSIipT2QUU/k7CBCqfj3Mv9iCkVn+FvPKc9F/HNf97Z/OUhe1py4zPNhW/+X9ZhrI2HHg5IYWhtyMOm45Ztl+Hh8CjaGd+/i+WzczUJjpwC0v42Z83OyDdWx5xvbYSs548ICbtdAYlvip0YvzdSI+NRr6elNHKyI1u9UbUOjidxg/18pnA07Aq7wvx5p3JKnKQtJRCfkwIVl13lljbEsn8lv7tZ3easYVRmhPVj+PqYX+nrdeDqkTTXrApUmrNcgKuebeOA2lKfbPBszj3wgVRND4bd202auGgFZlSiHACJlczd4aKTlFBOGnB5hu5cApq4raFRpBdUc8bXCfmkiI+/pk6e3vDsjr5B6sV4K2HIr7/bn5ydP8zrRURAr6qi4dUbJ4Phvm0y5n3K8YokO9V/4aT5jEtGJ7pqT7TACGcQ/4aIYngderDw8okliR6waGGrIVmaIzFnd6YqbgSMZU+h09PYZIsxVtH49Fp271kdmfSpttjzrHdkomhnGWs90rWzad7o6hX9/5uomudenQsLv2WGUacZQyjl83ibyPadzPjvy/ilv1YXIh/MifiKZdNTNyO3/C5dlmEMcet6USWIPxhjC+Coz+RTp+ipGH1uvsJqir8iY4Z7QavOUZdJdM/6Qnn2D4xKpWI9LOVYzovkeMV6FGMukx7VVYzvKrfEV7SgrkJD8OeQ/ucdTkfvqmgRoX2JUhsbc6EfjHXdzahtZHk2XIxlJ5+NO7W2Xc/V6CQlyQn5DFNufAyn22bcT50vHF3sqJ7W/Y6Hrpcbt8QTmdnCa0PV/9xn9XjG3Et2NPRNXIG0OehrQ5mWJ/I3bUa0n0BhWjRrcdqdmTi7I+j4yQ8ZEdaeH5sLrM+TUqAHM6P8K5zviXuQFJ2simJiu1LvPh8NH/AlJapUQ=\", \"config/data.json\": \"eNqNVE1PGzEQvfMrIquHBAEJ4aMtFw5FlSr1gERvkFoTe7Kx4rUX25uQAv+9Y6+dDwpS9zQ7fn4ez3ue54Nej0kI4DGwq94z/eYEl8pRhg0XUFUah8o0bRgKJaS0fjw6/XrcgHtsMRw36I6FBu/Rs6OOYKY08gZCQGciyeHh8PAk4wsmgKswcGF1W0eQabV+b4kLMFJRSUR/1btnP2GKmh31mC7Bt3h4DEQJOoIY/eqiSWZOe/jM2ZrHGg3UyNWM18p7ZSriD67FjPVQN3QNJT8oZLMez7lLP70fN/Hnu7arnZCicn7lbNu8z5cAPeItu/OemLqzrRPEfhspb9AHZSAoa3ImL99aF94CSu7W2WDpXDZJpJO9O3pOGqaGZB1o8TUimG+02jFGcKCiVqOTz6PMsAQd70BnpfzpRdGQitjPeERJmfFofFkyxBewWkeLQBss79oDK3BYbOLwsVUOOWjNs8s4gpjzUlpUbFNv47BxVmCRM9eNTwQWKnDpbGl/anrpQ8onN2TX7krC+r9f7vnDg58MZqRNv4uvlRxcf2JH/4L63okXnzQZFHBfNS8NaTH4aI/04UVuhfvvjUHVtItkfHfVq1ppDW4eAgH2pDdtjU4JbttAL5vLsG6i/IyuCOFsXPpvwHBlZryx1MGkFLVYg6BOWa+CWpI0RnKDFaQfwiqjwpqvVJjzuHuB2KRgZh3XqpqHaloXei/IP0kqZqxBtlGyK2srobOrzqY0RmJ6fDk+PT/PLMLWpDyJnlzI/vggt0w11tatt0xkkDSf8vvdEGf/X4zoK+MiV0vrsOB1q4MiI2EcjGcnBVTDE4clKJotkQ5otDgQmxfxZbSpBFq5+5qmIBZo4ptgEsl0NTWODCA4VWhnmw49ks+RTyGQ6WOxb2qc0cnc0jRx1vvuVXC7RKeh2Z9nBZhn0IfwWO7B68FfIJLkyA==\", \"config/data.smoke.json\": \"eNqNVE1PGzEQvfMrolUPCQISQqGUC4dWlSr11t4gtSb2ZDOK1za2l5CS/veOvet8SCD1NvY8Pz+/mfHryWBQKYgQMFZ3g1de9htCkeedaryCutY4JuPaOJYklbJhOrn8fO7AP7UYzx36c6khBAzVWUewII3CQYzoTSI5PR2fXvT4gonga4xCWt02CWRard9KCQlGEUti+rvBQ/UD5qirs0GlS/AlXZ4CWYKOIEW/umjWM+czYuFtI5JGAw0KWoiGQiBTM3/0LfbYAI3jZ5B6R8gun+75mReD71/T4pu264OQo3J/7W3r3uZ72B2bzY4UBMEOZ7kMm04mE07+TYgqOE0HZYseKDk5ufg06RmeQacbyHb7l9fFYQzxeCcgqsw/vSk7zBex3qQCQhut6MTDGjyWInp8asmjAK1F3wMCQS5FkZb83Ol1Hp23EovZvW58YbCkKJS3xZxsSfEh7+da9T2VsznH2eHv7YN4fAyz0YINHHbxPanR/Yde5hFoGLzcBtt6iaMCHpLbOuvj6L0zKsStYs/IZDP/+2Ckhk9xGd/MBmpIa/DLGBmQ8+XJpm3QkxS2jTx3QsWNS+Wv+IkQr6bFfwNGkFkIZ9nBXCm2WINkp2ygSM9cGqOEwRrygrFkKG7EmuJSpNMrRJeDhfVCU72M9bwp9EFy/+RSVcYarHaV7GTtS+jtumtTHvK0fXlzdfuxJ5G24cJzzXMTVn9CVHuiBhvrN3si7o/8efTDteM9aP8yyr1WTsNKNK2OxG2E6dO6uiigBl4EPAPx3Cc24LH3IHfzcLufJWjV4SzNQa7QpImowhO38q7ju5WYQ+QuT/IYcr1XteC7hOXZ9jaEbgqEfUavwR3/LgXY/wjvwpPAk78n/wDAF8Ie\", \"config/orchestration.json\": \"eNp1kT9PwzAQxfd+iigzEfkDEu3I0qVITKzW1T45pvEl2HcVCPHduSQF0YHBkq1793vvyZ+boihP4P2A5oSJcCh3RemEPHn5QGofmrrZbpvbQYCqs54h+J79MVYTpDdBrnzgXo7VuS1vZpgDhoxs8ijJ4j80G6xzY2719cuZMFV2gJwxr6Qpja9o2RDEhXOYI7zoOcwR9o9P1fNldX8dgSF5TRAYE3AYSZebul5G2fboRMvGQMLqtCu6dZLQIrGZJPfGCyT3R3J3v0qEKJA3PULiI4KWZFBWr1VnWdMtsgjvIUo0WiSruwFmjBMviqa+ljB4ArVNqNe0aFbI5U9+GBwijqKGaEdyS6aurevN1+YbINSWcQ==\", \"config/report.json\": \"eNpdj0sOwjAMRPc9RcWaRSifBZexTDBqRJNYiatWoN4dN+Wf5cybzPhe1fWKE52dFRcD2LYPV0hxyKtj3eyNvvWMeByd7706FjgBd1Ego+eOCmjMm6SRO3QBNPFFfH7KLfJvzeZlMSXfC5YhiZhQSrR4EhkuqvSJMkgsC9TcLskTBdt6TFeV7irMEoptIbsblYpmt170AbWEwQml0rSse5qeMGuDpyD/hFFiWi4gOpejm0M1VQ9AamKs\", \"config/train.json\": \"eNqFVdtOGzEQfecrUJ6bNgmlD32DNiBUCohLW6mqLO/u7MaNL4svC1HEv3fGXm8uIPUhijJnZo7nzLGzPjg8HLXW/IXSM80VjD4fji4D1+Mf+LkUzcKfn34f33D7GMCPz4VfhGLczUbvqFCZCuRQJim7KVTC4LkFKxRoz6yRMaHgDqTQwEqjPQZTogOoEJ1NZp/i7wo6Ucb8sg0pRQfFCmMctQqasqeTSWLhVq6Y86ZthW4QqLl0ECGhCi65LoEtuK5kgkfaaEhNa+A+WGB4JhxeGL0LBweMS8m85UIj75ND3NsAW5O33HJF8TXGMBrPiETMr9o4QVNUPrZD0BQksugioIL0opTcuQxLHEVTreWeUibvJ5Npj9H8iHdAXEc5qvgzq6D1CwyOhyAetuKeM/zGkjoqmym4KirO5DS134vOYnS7T0OTe8NcK4XfqSHqQpBis+PjPpb1rC3Pck6HgoI3DQ33HxAeiSaTgLeixMDvJBeTppGGFOv1Y2CtsaM/fX4yzqB9dk+vn19Y4BUJ+HHI92BxToE7K7eWS7MYW5JNJXsSDt7C0BAZGyz32sxbw71G9gV7IyVu0qJ9jXoD7cAWxgm/IjUx9JLuD9Y48BtXthbQqFAlX1TCkjom+DZ494FiWSaUPwi6EbwGpkAZu2L4NtRC7ktgARXgT7FjVmA4QLmActkaobfOgD/AdlymC+ziDc6Soa+ZUCp4XkhICYw43S6pNCXWLwHavZzZwOzAuWSu9eBToXD5CxNsJJ0NpqM3gxWA28RZhQ4+9dr1eDotekc4tmmug5QbzqMNHWgaodo9d2il4RXrxd0DIwn5PLIfD4vwKD09l8iKj2UUbHP0IpRL8Ax0R5u8O2KnD1++ze/zGnHftXjegm9u52cXvzZbbshsPXzy847dzs8vrq8yjsuUBS+X7HXi1/nZycPlfS4YNMCL2aT3dZ0fsw4kVV1cnV0PL1z//8Doj8FU2bQHLwf/ADHq8eY=\", \"config/train.smoke.json\": \"eNqFVclOG0EQvfMVyOc4sY3IIbeQAEIhgFiSSFHU6p4pjzvuZehlwLL491R1T48XkHKwrKlX66s3NeuDw8NR6+xfqAIzXMPo0+HoMnIz/oG/S9kswvnJ9/ENd48RwvhchkUU4242ekeB2taghjBF3o3QGYPnFpzUYAJzViUHwT0oaYBV1gQ0ZkcPUCM6m8w+pucaOlkl/6qN2cVEzYS1nlJFQ97TySRX4U6tmA+2baVpEJhz5SFBUguuuKmALbipVYZHxhrISefAQ3TAsCccXlqzC0cPjCvFguPSYN0nj3hwEbYmb7njmuxrtKE19YiFWFi1aYJG1CGlQ9AKIll2CdBRBVkp7n2BFY5iKNbxQC6T95PJtMdofsQ7oFpHxar5M6uhDQs0jgcjNlvzwBn+Y8g8MVtKcC1qztQ0p9+zzpJ1O09DkwfLfKtk2Imh0kISY7Pj495W+Jw7XuicDgGCNw0N9x8QHqlMKQLByQoNvzNdTNlGWWKs54+Bc9aN/vT+WTgD90U9PX9h4YDXROB08A/gcE6JO6u2lkuzWFeRTBV7kh7ewlAQBRsk91rMW8O9RvYJe8MlbdKhfK1+A+3ACetlWCUBoO0lv0AY5CFsZNk6QKVCnYVRS0f02BjaGPwHso29tksobOEWoqQXg8+BadDWrRieiLlU+0w4QCL4U8pbiBjaqBZQLVsrzVYn+ACu4yq/x2kZZdse5c2k1jFwoSA7MKrpd4sqW2H8EqDd85kNlT14nzW2HuQqNWpgYaNLRWeD9uh0MAG4VJxVmhhyrl2p525RQtKzTfLpZFPxaFMMDA1Q7wkjtsrymvXc7qOpCMkdtgVKlhWjq4lV8WYmwjadiVgtITAwHe3z7oidPHz5dnpf1ohbn8vnLfjm9vTs4tdmyw1proc//7xjt6fnF9dXBcf2lODVkr12/Hp69vnh8r4EDCzg+9nkM7suN60DRVEXV2fXw6HrPxOMvg82HXPKcPBy8A+2VPOS\", \"data.py\": \"eNrdfV1320aS6Lt+BRYPu4BD0bKd8WQ4Q+fItpz1RpY9krPZORpeHIgEJYxIgAFAS4yu9rff+upGd6NBUU7ycnVOYhLorq6urq6qrq4qhmH4qcpWaZUFiyy9Ti+z/TqdZ8Gb92/2374tz54fPPtL8CmtfllnTVCvFnlTB/OyCo7zy6vmh9cfhnt7n68yfhPkdZDWdX5ZZLPgIoNmWTDP0mYN/07L4ktW1XlZBEbv4G3apDVAnlbQDl4O907LG4ACPYoMOgQX6SItptlsENTpcrXADzcZ9sZPAKkoq2W6yH/NZsMgOE6ryyzIi9W6IRh7q6qcZnUN6JRFpqdRlTfBZVWuV0HaBGnQ5MssSItZcFPlTZMVMAeez369yqb5PJ8GQJ+mHu6FYbi3N6/KZZAk8zXOK0mCfLkqK4BTFGVDc6j39tSz6hJ61pn6fjlVn67S+mqRX6iv/6rLQn1eps2V+lzW6tNqkTZA0KX6Xmmg9S+AavZCf22q9bRR33BujPG0XCyyKeGnUH5TrosmqwbBLJun60Uzy6EjNZ6lTUZkkZbq+4AA/grE5HYrwBWmoZp9QtTpRbNZ5cWlen5YbAbBexgqvVgAjA/pCt8OgrMM1gNWVxOsWC9XG6R/sdITh5VJka+C1Uw/26QVLiI+TJ2Hw5WsMr78Rb+s102+2NvbO/t0/P5zcnL44egsGAdR2FRpXoSDIPwCXDSj9cNvTVY3Ybz34fDsx5ffQsNiNVznRfPy2+jg9p3zF+/9cHRydHr4+ehtcnb44dPxUfLuPfzvzcfjnz6cQOcwYd5N5jn8L5+F3Q6nH3/2tIfpUPOjkzcf30Lj48PXR8dmu0V6kS2AJ/emC9h4Ae5k5ngg76d0XWenSOEaNkt0CmsNa3cENKri0V4Af8DOp2mOuyOdw+LAVpitaYmCulxX02wf8Q0ugEtmabUJbq5gbzSw2X9MLy+xEY4D+xl2fZGl1WITlLBjh7RJ9oClgkWZzhLY+PP8MkJOGSFzBv+X2CQO9l8FyHDn8GyAHDJhpLh9gu1hitiU+sb08iaHp0aLYbnKiiisYMmAj8oZTHscrpv5/ndhjAxwBcyzyBgw/uVzq3e9ns/z2+EURNC8XMyiOBgDVYe4G8O2U4sVIITvhjixiGHHulm2qDO7U1Nt7AeEAjPkJl0urHfZ7TRbNcF7ek2rhBOAp10QFS5aYC5oFP7j8MOxYAlriGwMIuKXdV5lwBYbfPvX4L/OPp7AUmUzWLASQAPvw9YHCs6AeBugGG1dGNI/dUR5iMoh6cwf6ArSDzghL+oG5XXEvQa0xHE7BUb9v9PFWiH+xsa5BDDLdd2A/gCRGpQX/wKhFfIoMqEZ4HIXzlhz4GYlYY0fVib/44Ny3YAuwE/LbFlWG/yUrmfQ+p4gLnNqCgBroDrsEjXEcJbP51mVtVOJ9Uyl05ZJzcMPAthekVoE8Ci4EyD3amrYoAY8zudA3EbGPJepTc6LdJlNYtK8+BF0XGBIsolCLS020RfEI/jbODig5vwV2vMQMStNVjPDvJ4uyjqL6vUykveD4NnwYBCkF3XSlIvxs2z/2fOtC0gS9GkrPp+i7FQTUku5Kuu8yb+wnoXRgqYMnoWapGq69gJOhpdZE4X1FGDD1zj4N9idBSifcBs+n69AIGkD4wKYBHpnjAlOHMFlqGmyCjS72Cd1iwuIeU1+4Z/JeQiiuE5WWZWgKRDCUiCBt6HBXYdWvw45NGMDDoVQQYRn2pTLfJqgvElmoBZBCm5w342U+mwFJyrwuskLIv+oFa8nQCpG0XiP+jErmuHyepZXEX+px5+rNSjm7Davm6S8pq+MWpOhQELxP7agoCROWH5G5nN+FHwDYrRZrkJDbGtIIrRvdhXaJHJNEgykCfav0QhL62mej9+lIIEHsH4gzprx8wHt6eQ629TGfPCPew/R3Mui8J+FYFnWQ2C/RQpbXuNqkTaWpZnBtkFVl4ipQXq9jlAiJUBUU80N0EQC3VrQQ1qVBdD4HN+JwiORJ5pOgbCkKjYY0tLUUWcnvoOhT8rmHSppJXyUUQ2AQOCA2AtmZVYTLAID4gdhKtlD2LcikDQvSg76kLNMHl4uyotI5hIjZiuWHzT3KLYQJoC7IHpSapuckQChNL0CAX8nI/1bdR9Ae7BNLIxlv1AfWZO8AHENxsZivSwi/gfErDIwca/ANlHSOJupNcItAs+BIVB6ZW4XWrC2Jc8JTQVWQvBGxooNI4Jgq+dESf6MtBTM7m3BRxhpes2RRACexyEZyABVU3Ow2LRtuCeIvxbZLTrqjQaoELxrx0DK36TMNAQ2bEdS5MfHQpGqpSFNsyXog9NSTR+eFeLimZmLjTzAlsIcoAJgIyR05hMeqfuZxMcMVptJu4/x6+/MFRYxE9zHPQRNp806XZDNYJMULQSLmpbdYMGemIQWgKii08Ui4g4t2S0o3Db2LsW5RsLTZWKu0flErRAdONQBCcwwOO6Cfkz0sYFIDqqZB2xP/TB9q7EW4eE//4nW3lNbZACEIVq5ycUGaBnJMXx4sUivs+cXkeFNINUEYEQxoSV7CaogqeHt+Lt4yF8jeBFe5GCcyEQSstiW+S0cFcn0Ai6CwyOeoKp0Q7Nov/JkfoU5cNNhWsPBOYv0aRNF1mpjqC/SpPA6qyowtEF9oR4ah/klYA7WRLscCDT6Nfg/+L9Xr4zz64uDGBjiiXWiff3uT999++eXb5+9Ofr26E+v/xI/DOb5n7tg/vLt24Nv//L69bMXL549e3b0uiMx/Ag9Q0j/HvBZW/EDUjHBFQANg+uFKyULZpN0IKamsVnJhIadXGco6GEUL9lhjYh9zBVj8H3LQAZGDFNokcchRFqtixz1OIE1IBAyL7+Ng6cBm/bPnzyBr6J5YRVRJNILnsf5wYRfwmGhZIFJrb6xWz2bWHwNA8HxvMoiwuJv3GcQgCHvvmGwYOUPgudxbCAKE/pOcTF78cj/YKxEJNuTaDoI2Dvx+yyGTKNv2WWoRyyMoBorjBgBNT/WBAJ8XoGAGgWr2RANp3f4bRBYusJnFbgzALFaFvmUBDJBHC7K6floQHoissDFEzWREMDR0WYI+C6KNAr/dnL4yhZZgBf6roaIbcLesISPxZEek63eW2UCT0EyXJYVSioSHcOmTMi1Fs1w2LEmk5ZaGhK7lFh0jfB0oUygkTJZAJ+8BlSpydaToToEN+iWbQKCTCY/uR0zZVLw8zHpRwY6RLKsIsuopFbbRjtarprNQ2MJTem1dpsd3YJqOmbv9yF6B8pKO8je5vX1/kU6vQZZkWG7gPwHQQlMlBUZMBe8ePb8u/0LeMhuu+D925qUKPuYRaaQW4zPYkDvJIfdmCQgPxbzAbpXUzyoirrjg8MF2sK4Bet229jWj9Xtkec6oawNonvI6A6zLuA4fR21UHAKQ5ApBTs3cCHZI60eRhaA3o4wejZd44ns0+nhDx8Og3+BTVAARy5BHIx/PjwOd+9ab4rpVVUW5boen3w8/bBTZ2vS4ZvTo8PPR8Hnw9fHR0GOR0o4scMRJbqGrRZ8Pvqfz8HJR/jvp+PjgXq/CV4ff3xtPOdrkfcnn49+ODo1nof2WJ9O3384PP1H8OPRPwh8CxA048/vP//nx58+B6cff37/tu34tfM5+vBJJkWMnBCbBZE9AwOhXRFo2RU4AH0o7QOL49zGth/Fv7UJ0yGzVWL07XGnaJzEjT6tyrpmRQa4HdhtRDJvayJgZmt4jaK1TtAMywujfbuz09mMUZS9jetJFuzA4KKRvg05J0N0MjBUX992z7ELyCakL3yMWnDttG+u0GWP29ymKWE04jNLA/NQ407w+DB52GuNdnzCXsTiMoucRYy7HfSgw3SFHuYoKrLbJlJziAfGGc9wf5815YpIQ66sDtQViGvroSgHnl6n+UUFQt162is43h4dH8H2eHf68YO5McJ4l+5L9LuG70/Ojk4/Bx9Pg/c/gNA5wn3/0QSmN1kc/Pfh8U9HZ0H0fRyKpLcHIn6UjbTT/qY9fgaTePM5ePPxp5PP0ZO4M5vg8IwHcwQQdf6vj+9PTEEHba+L8qYIPp7whyGy8vj74PDkrTxQ8xnzSmsZ4oH+838eAUW4G7H63159Hw467ZT4w2nrDRHHdkOwljIYDzZHFGtzWS8Rnjb/fybc+I+hG+wk0mx4A8YCL+xuqD6p+s2YOXZrhz75CZ1x0Wxx0LlM2yKwvaPvvvj+fdsuqKOSRVTHvsUSPvrebP19V6h87fptneC0XC7zBgwzrYtQFTHta1FGOxzgdtBDLauPg4jjDcD6nF5H4au//z3kWbTnL/zGI4kjir+wZ5kQcLS2oUIVJ5q600TRmSyxhpqrz1Xw8OTglAwnPbnxp887OQXi3agjxOFTTmzfz/FoW2hBs9uFFFVWrxeNkIG3S3oDB4JRcFGWi96bd5GeuFVRz5JzpNeMMo0kHkL3iVAjG8PSlWOfrWXCQQBVY+MAPBu+zRrgcbwW6EHn3jx5ASmXMP0cv3PAUuiOYCLsGaCD5j332A5bjpZ3ttm9zJqrchaOgpBOj4kYsasqX6bVBm+nHGkgLA/7wsQgQT/fIkUXAhxoAVwPJXphaeFryV4vuB457YDuEMmPYqdZL4a8NiDJZGEQgsUJvvE9nazFtfv4AbdndX8vPUYNPe+6Qp8uwOFAD2arXPAn7W14ktdJhp6J7dPpA4R36Y8BYQ78lXDu26/3rUyROAFYUo/Q7KgiahxrBwvur2qZF3D6yKc9jhb6HqyqspyTXMTQFKOXVqj7TbnPsmO+Lng4xpLCENH4E1mgjSoA1npiKEZxcZNu8LIRTiaz4GJDQU3cNctmGd8iUWM1BN4lw6ELnXx03MjaOMemDJqbMlDxIipIchhgHAJBSr+UoORgQiQ69i/yxQJA7mMA29nfj3O6MZtlt3J2W4H0zqov5DcDtEhq7OkjgUz9cp3CQazJsqEi3x+u9a1btd9Z6Xph/wYt5hfFeveTU6xPTFtclzBDkKc4UcyQEJP2iG1TICpBePCg6OxruV0i94ImOoVMqMjUxE7r6VU2vf6tUq1LzV2FWU/PXWVYt/tXiK7uZXFCYaQ4kMQIgS2bLWD3rNLhO/xEUJC01tUFvEX7sMaYCET7Mqu4Iz0mG8hsQtcisMm3tcExsrQwmygcZ1W5SqosrUErRRI36b/OZoe4He+w7XID9iuwxTKl+Z7RR6aqFZvli0LaI8Kwd0lfjg+MTer7Lrs2u0UOJ2fanXEPYERmWrayhQtHiSkITBqZG1jL55OY79UlpISC7KoMj0srDF0hmCAjF+llPYbnfAZ8c3h29PCYNBRebycKOA/IDjXmI1pNnBavYkF3TThFc+FxIPqOAzH9JRgAHyagqVwI3Lq3H2IGK2vTGmEIWIl6G7VRDK0j0Bub4MTQ0lsw3oW77PsCHPuc2yDYUO5k+EloBMxqUOZCw5hqLR+Aq5otNug+XKxnrM2tmEvvcDiGvQe2D0Rtn8KWvRGrAmhe4cYHEUe5BYABUs3KoCARjVvcQYH8oCqUaij/qoAr+ozMppiVI62oaZ2lFZwDJYQETta4meP+wBgFbsxD/vYRtxJpHrakeYqByXUDyuupkISDQHQsnIrvundpAxPpkb+4CdRw8egBVAqwr1SEp4owo6tPOGXX69WKIt2QV1R46Eh2lzkK7U0LQ9cnpbaR8nELoawYNWkyYCwHplQQYQ5ShSw0iWiO2qA8QzRTwOCAYvHUBaGAZuNBjDvmfY903us1mMRWpCBVCvgf63BcFWOtAnGdlvpOdVpWMyVL7CEMsdKUTbpQF0QHxiOUxFUmzoMDLYJU+KETTajSO8bB6pehrC3GFkb25eLqalPTfbZxIyVdh2DspTi1IbCIc0HFkWJg8Uk3MAQjC5T46BxKxBQS2aGkGTyF3GNj1Q6a/5ppFDEtoUmbCC/BKdioRc6g4DdjG1WnjUFSbAlQDIOHVkuxbMdGbq7CUSCRVBJX1ZQR8h0GiyR41XYbuYakNS/sb5HMsWlNvNlN0H53DVRR46P+9VMxFU5P3GR8MnE6y5rLy45FrAnHEWKIH1DPsCt5OUS4gLhoqvyW28oSGqv0xN6jHC5FcijqiXg/DxVkjh9PqHU4iYdwTly27EABDElazNBz0jt6pIfj+J4Xz1s4wTfBcxOhNiakHYrHwgDoJcY5JCvOFGxH9JLhGw92bIGBLkjEf4YKVAU+KVpIjgYQYYHh+5cXy8TpEk5clOiYAaSjlkKDXoSfuCgwNB4XJQpliQ2/5BVGKib8XEId0gWofxJROAS/GqZf0nxBGVNPeueyTG8T3S6p0mUyr1I60cBsLF3R7sVQ6QTJmQQ2fNbyYGhGS6qj4yIryOlusnTI3ODuuJZHjKYWp0Ij63t3bM9GcUWP0aldER/HQF/fY29/vaJ8cFb9+5bcC0Nzl8U8HljWewNUL3uOXAYzOuHKM4nUQAYj0YvYad1yDfVI0iYRvrA763YmAOHXdg5qVHlhLinmdDUlEh8WgeJfE9DAmk79NMHojS48lauK5JCPQ/XBVB3hdLW22rAgLKuIj8Lq+TKdXuV4aWmyK7I6dBZ1xm/utcfxDLkDrIPmZ0zyqEbdyCvLedkCNgysvTYDjlmasmvQtDFaG9k9bIK15iIGUxGhMHnYMcm0Xdcx1MBk4mM6SBo6rbdXUytOqASCY0AaRef52vY5aCXRBP9xPbd6fmT86W9ufzOVaWxP3gkGWqM31DqAknFoxliSbWhkHkfYIvbBUcaY5Cu7sWdCX2coxw7dPhjam3w7IZHYaHhSO3sVkS3v7kVDunFyLTLnBItsXwLC3gzbncDjTLpTQRF5AcQzJnxH8FjGd0aJW+ydtMB7h1AOA8EAzhPDscvWITtfZ…6103 tokens truncated…5zVgBRNxZ1sO9ahCHdDJYsIGuy3/j1nncgr0amXVvRkABR4P8FvypzGBlQgVtIN20rEBZcNK3PuuqQCtmSA8GRst5kFTKbkij06JZn5ETBjQMxwIqLF4w1BTetraJrP4qBUPCqWU3ftenix8G5KIKFIIzw8Qn+q01Tb3GrTL0EoAYPTIAm8SI00oqLcrvlHR+OJ7KnpxfdcVDb4I1GLOdP7JIfxJrdaEy3+qiO+RhDgT5PEFmj8WnvoXlNC56hbA3Ja+PCB5dBXmQLOinBrfzSNtoK6+y6rOEcd23fCZAGQWgsZ1aU52eBBxicK5LwoNMNuAU4yrpsesnvxDEDbjChdD1SniXs1BG04h3pprFeYLwH9YFdCtzTX/wle/Lt6Wl8ylazJH7JvoVJs+8I12jjiSlqfuI5xMYUK4CBLAYs0kD74vMOFP1TXLfgGjDUgXv9akSSPmurJJTvwCwkHTzU5wAfZw59ybq+ScuCwsmSKeOhUKpHrB2CCdWZOyMhheFyjYnTgnQHPuiwg9YmfK+itgnWLCRSMAqGZjAyVESOjTukjFa5M5FHs7t+hvARnjmIaJZjF69iOy1NQoFIgXMX8QRkyVaPI0CNcGoyilxfs83Kqu/ING7A/sH8g3YjeHfFUWLmo/a+uaRR8/GWgYKh11iycIA0sxHsqc8jBnpqAdoDGm0A2d+SAZFRd0PNnV7shZOuCdgkb2vIBUrMsSmxkJhaw6kCLwaf8WV534H3lMCsldtZoAcdoZ17MfEUzc6sfJa4onwwnQY1u9GIbpH2tpeiLDg7jeMbhfHWNz8NrI2sA6VQHphimggnlJNJGPsYDfuGYnExl6FhCUDT+GQLK5oBw7O7nZIjh6zqIBgcKFXCJAaLKRODQCUUEYO0fGHYDbU4UhNgSsz7gDnwDWt3wdb4w/TG1Btx034MI/CP3Ra/hl+c/H5SnxSrk3+cvDl5/0V0m95g0RLjP18D4I5fn62/O78NzJ68uSq7tqEsBDLxDCN9WGHtd7Gp0yveCZI3uSnMQeQOORbDCcwlyzWv2+4AJ6GqlBi2kD3UFWo89KTgpksHuQPm9KZgmKZIi/2Z0HNuagqNpqx4N7PKTHnLxizCuvGQA63rKwdYj8SpGUtTP1vbHxzoZn8EUJV2LsvFMUjNlceg/uBxlu97D6Zrc9AaUNxI1dp6HOqvHSYxnlDaC6r/AAVIDbJ5RKSO0A6FGij50PXcO4jdQTx89asMihOPbGs2qaNnASla6Ix4AaSVsB/ErHRzwPxGASsti2nSxgat7h+7EoN6bwpAnSqqyI7DRdmtqbxbOmXfXLSnaRUw75l2Ir0KUcfyAJqtsg2k3rVCOMIM3J37OLCWEKi6goxQz87br0otZL+v+BlxSHz69S2wD2arBcEemWMJRjBxfQn/QgWMvlSQKmCgBCrS9lJphpWordbQU/hJtp2N+z16tHAuy9HExMiqc/bj6mnNAtOpQqQSBu+qn9bY7nLR+cXUmpGCOgAztRVs2UCSHMylSrbWmgGaqbnWjGQ4RbTtMoPncewSvMmo0HoIAP8TJt0pcl0Ud0HoOBefnj4+mjA63+5NCmdGZ9YY71DxJvQ0ORqdWF5lQpBhI6hnGi6oE8QA9GExTaWQirezwHU55zajHNRU930cK3mkVFTrLzVoXJufXeCW1c6ScSsuHBAv7e7RPCgZlcI4JPADvEk1NBIPwvjEvIVYte84WnMxCMyMaHfoe0dyJeR1PrsPAfJKpGLYDbCfnSs3CBENFQdTc6hmgGIV7bA5SPLVvQtgr91CDmAHRVaDGwRdasotF9KOeyqmRp2kWEBBmXPMa7RsgDvc3u3mYYat4GJiBsoEv1+mEspXQM0vrXyFzsgk2O9ME8mgXyHtLOtkuQUjxwzbdEXYjdrD5Nf4pwBGykal5SPN84hUre4Yjx1lEipMS3f9gFRJPwaB8KYIJyBAaNvJ1AqGGqNpinjTNIqhVGmrK0gyYnXMjjaQAdFqxxI0ujs40WumdjfmyqNsOYPZ1S7D3zEo2w5CYGM3qOFbAIMMhJbBuYS6z0M99vVM2906FNMnOJZ4UKxCHHhJQCOQ4MhUe0tBieKPoB2vOhsWCYAUWFDGCZll12UHHfcBYaUU0WsCqy5/dsk1E6YbzK+g4sEGiOKQPNdi6CzPzIa+liuWlt6gZsof9PiamVIc+ROQQVabLL9MqqzeFBlTvi0Hii4g9VubE4jFoclTYz+hkvtyBO0EkmjcA0MvtHB64S/pP8yoLKS6lIg/qojqCyF4heIx0oJlj6ymUGuAF0+ZI0GVmqSQmMTyGu+6CveSTdeLGLaAmWA57tWnZbNtEz+NmDBEWY1WYEo/QihRQV3MHVr8C4bhfZbzUR/KJHHTKxREELtpHThBGrMhhTwwmm8pyD0OHdEB9ZntMZ+fBZOVAfr8eaTelnV2rXsJKXZNTEU9v/F9i/zm5HwvYLWC1erWVKxw9cqsNu3JfStKWV7xIJpw7Hd852gY+D4CoNJ9fR0G+W+Zv1CnYjer+BWvkguI6lJ2oQZdou+xXWPTswc6CBoyIEjI91gxO8aBVWMmk+AkzESOzYZIsJOQFmB4oW/qwxo+gRYJMMFIaEWNpq7HXNHqxAwvl6qLzeKBV09zV06v8a6nqizK75OvISV+9g1TrWN7T+BeLBHCtpf7HkuDVpogRlLX46BmkevZAWbi2geJO8gS5/MgyeEaKHGPwbkdOnfkPrgAbXGJWeBcxZwP8OLJGE48Gc3zBlveRdpCLtyVBSevYe0BCkQ0csf1pLJNxRO6TyQbco9UOVW6FVA+V8f8VE0oOeqetBvs3BVqJHShbIalj+NhXkKt1S4Z/WEYvCUo9Lj8GvxmdWDAwPC64UeFjzJJnuU7pSOPdP8fXTYGISb2VWnu+jY9XjMCYf61tk2Yl8z1hroHqLqShotxNXruqVdctB8bOgPv9sEVD85yV+oEPoUrtwYUXODI7447oYDt1M6Cj5EUSint9pt8TgJOezSlVAHM3KTjRgBjb5Dwz31lhi0wBj95d7dmnGkwUzGp84lVpTP6qipSM3ZntTocdexVBZPVVLku0XfNt+4i14VS8oHlir4KDad1UhRfVO0mDL7U9ceofnhIbmNRLabvAEhJluaAdUiHrPYuvbjjOmyE1tDxiVLXZjIjs0G78P5Ydp7ABhdF9USVCQ2LCZQTcT2P8A55QIcAmNiJUKEAm8QVl2golu8TvGY3DM1cHExuDo7eYxj+6DmHk1TohBgPAbTnB/UtVIEAq5jkExmO3OQGHY7eIJ5QBKEdb6z0zeNMlvNcCN4hpI6t73lXghb8BRKilJVpalnRcpVd6Yde1s9CHjsIUzinYTU4u/IrCKs8uk4wUpuu5JLcAipdL+y6gG5jUjAzXc0EUxQ2A0/NkYP0j9dyo50HfbBuxxLrq7c4CzByQOT1vIgdnlH1I9xNvB2xaqgPlAZOmKJgfVwgM9v7RqJs4yO+O9OXXBZ1Jo8ZyVTYVHWcKlMZ7rdQ5O5t1+JBh9s3jXrScuQSz8mU9CW9H7fwTvhWkYL1jwO+a9vL0Cu57R53BsIlU0RS4BtefQz21LQQyKFandA2l95pF+qOOG8uHJsZIOaUI9ES892zlXYyfJytzpPZKl2LIRmJxQdyvH0yEwA82JkokDw0LI8vdxM1sJwEpF4k6r+5aDxi5Wz1+NzNqY3wqQabDk+sVl1hGGxptgXq3JOygWR64D4hD3Z3mrV7Hf2AcPL+j2LfVfmXKcYueIPreFo2OYQ1wGRdolh4K++CDKcPBOfaQZ+xk+Of9PHuzX0dHOrivB44HN4qq/4No2P23OFT9/2tSg2ESqCCKfv3Hd10xZFOz2y3Z0ZrdTtTvdwFxZ685g3HJUAyMfV7fMm91uldPyXqBZvnLs5GN1RHymF8Z96BZBO3tHKL4rPAgOjM7Bgm9P/J6PvSaayrB3+pfTiWHHkL6GA/0rBJJs0f9dDpWAMomoodzL5zxTvNNl0i6CWcWUov4pIHvzt09p4+wEs+8emh2zQwVo+R7cxoo0qJ8Fa3LUb1stuhMvd4CtB526RXfu8m2HYj0+HHPBuwpUNPNlTrEvVfNNOo8rN01TadiXt+yBtFLOqYgh83QGpA9w582FFzIPmE8sJHRE4HNxXJ2ey2JjMduh1m6nwOExW6CSSwiARvy4Z1wfk4uqNdjt74h3iJO1dSj/3cUPMlw0cf5JLz/fCC16QuU59oFSCxn8a5gI4O8z9gmPQdvSz5RDxlDb+WxjbZx7Kq1G9k5tJkQLSkm2wT6dlX7PHEa49/waB8tqpO3KL8vthhHyGOVtqyD98DedTMFISzxaB/1MODecUACA3fLba9PFIKrm2qeTOi7db2JZIbl7LbYHRmap2+ZP/f6sn/s5b09OCF7R6on1LZ30x5OgBSWENRBYgG8t1r1E8qSj9PQXqnaBafpQj1blhP9d0UXUal2DtWTyGn91Nr89pCkGe2AM+7ix5ztbc0gzfWeVdS1pekadHm+BuhYWWcFQVuQ0vwOkc3zvA2fJv1lUx0K+0R+RrdkbsLgXd3v8Ku9IALG6J3LlbXBaNVgRoVj0Ds4u7NAWKFeebDdzxyfYVHdNjzhJ7fPhiZujlYOV2qlWxX9MMQ9Ugpwdygw59e9Nx/DKsxuueuVQGDn1KCoX+tvNL7A+h5/fK6lKEKyu7qCNeDVppfhtEPqdIUsaVpYH6QgagX/wUtyJyk\", \"viz.py\": \"eNrlXHtz2ziS/9+fAsetrSEzNCM5ceK4VqnyOM5u6vIqx7s1eyodCxYhiWuK5JCUY43X3/26Gw+CD9my45mdufFMKSQINBqN7l93AyAdx/mb4FEiypJVC1HG5W4B92t2GZcrnsQ/8yrO0pLNsgKfs/fxfFH99YcP7JyXIolTETiOs7MzK7IlC8PZqloVIgxZvMyzomI8TbNKUtjZUWVLXuVJViXx+c5OfR2sSuE6R/O545ma86m++leZpVb7hb4uF6sqTmTvOZQDId31Z6xGD6p1HqdzXX6Urn12zJOEnyfCZx94jk999kX8tBLpVPTwGeRrvGK8ZHlS6efpapmvsSzNdVHO0wgKsF4k+7aITLMkK0rNxvts/jErlrJWeZEIXqTBUlRFPDV1+Grqs7wQ07gECYZwAWyH01VxCYwX2VRe7uzsHH96/+k0/Hz0/uTs7ISNmLvD4M/502Dwcu+HPceHyzf7+yeDAV0OBq9OXj6jy+Pjl6+OXtLlyYtXb7GCarr/4ofnJ69Uffyjy5dHzzSV/TfPjl79oCocwH/Q1Nt5/+7jSfjl7J/vT74gH84uVtiVvwH+HsL8fjg6/e+TU1khw8ISf/4Xf97gzyX+fMafH/HnL/jzGn+eQOOjH999Cd9++ngWfnn3PzjY4WDn7N3Z+5Nm4RB6+TE8/vvpP07Cz5/efTzD7vZhHDtHRRXP+LRCHTjn0wso1+owHqPW+KysionPPmapmIB0IzFjIRpFiGroop4dknp5bPc16tMhyexrXC1ICYMsF6kLupRFoFkjZ1XNdg8cD9ViAQqSCFkf/woB9pKSegdJxiNXVvB0r9MsrcRV5RarNIziwuq2WuXAbxRPqzFw6yMbwHISl1WrUJfKMVEx/cRpNZGMIHHoaBbPQRTWQFWn7Clz5GMHL+vaAdaCGUEaC+giK9YbCSjNJgqqrt18Ka1wm/5hokQSqgY2jWnCyzJM+VKUQGeMF4RaeOEzgKSUlWBVInJ147gSy9L1fHYh1qOEL88jzrDsEIXj4tV4OPG8CZGfxSlPQigsCM+giyW/cl2sCYaZFdHYMQ+diUddywfYsxoz9AXzyldJNRpIrpUGuLVKGPn6pky1rgussdaFMK1u3XrsLLMIJIWVgJ9N1YK5qFya1DgCE1MyD7CVZzVqjV4+MGpaxVUiXPQIh1K5qGt1LWmrmxYdkjQo5Go2i6+oCsjVcUjD4UbqZ15k8wL9Ez5i8aw7FQgCAyaSEmbcYf9mgIuFSKu6yui61ebGIdJgWwUHutTqWrJxQ33Ia0nTceypmjnXONIbbEHjpCs5Srxs9TQC1q71EG6uqUfoXUmurNaJCPlVXLr4c4gOJji6gmll8wKldp5lCTB4VqwESQVBSYoF6wdVPL0Ic17wpSQwcs6zagETSWZSxj+LURMwpdahclIN1E0iBGoQXiE12dD1apCikqCECoVy5u7zfa/n8YKDJmF8oCwSxEiDMFVlT1Dk4ngQmFJBEhhJH8GTfMFHg+C50SysIYUElhaJK6UwSw6oq2XzlsMkkXCayHdoz9q1YcIhR+wcsobXHBN59meWAHY3nniT2gwcwzG0t3xdo7VV7k1QCMit1CS7iRwQe/q0p0uvj5TFxpIXFwLHoFxpo39V1u67XXWvQ/BSFGugOQiGe52+UJXg2bNgryWNr3EECnfIhsGzffnoRk8exH7xbE0OEzRbB1jkY8EVTXkl5oBqChmmyh0fso6D/jdpfUv9YWymSQ1ToNfYHaq17NY8IthUDYilmgOjbSW/FCGAIsSwrgL9uTTJt1ToG39pnLEsQriU46DbCoMJaBgFb3jF3xbogXZsBjYOEtQZ/1G1y8tQab2t6hBnoSjIq5MsDzWvwCKipOU4VaEEMB1eNmpoz2zTCJYX8BSkhChajqSxCjDeKswu6NazCW5bPcc4O49mPo2M5mlk2H6KwIpyvAmgnuP3Pohm8EAPw3oA9MwAABMBgsKEr7NV5XqmGCcX/nWJiyiPR88GA5+dn2dXIOQppD4jp7LAq9EEee6pSbzwCOZ4dO0cQ+SC4AhTjpZC08icD1lkFdx4tX4EVRYC366WBYZkMNcjM+sgAogJqxCUGtKIkfPn4MVMMYcqOU0ySJmAPVk0n2KGkYipHrM2P1eLHYIPow+1vbUqG26wttaNdm2FqePuhJp4Ocq+piVfQpDqPuFFwdeAAGkeQH6EN1YIWxf6LAgCpcyAY3NSDwQ02X48mBi3oh7/ZcTaUX4ntqZeXOiFl0THveQJ+lKECrokF0g9KPIpuBGyI2i0SmNALWwOWFfmfCpcUBrV/S4b+h0GQLcg4RQjaAKu6sVzryGyDdyMVaeTfrakTDGRDClRhOBV5n5lMzPwtwaYXgQJfRNp4jUFNzp+83sC4HZ6Yik3PLXRz9Xxr5LFT6u4EBFUstxyHT1jRgjBEua8FJQ6EDLFabiEsDkOk2wOik8ZI8ipXViTs9uIogCP32hhilQ9Pi2ycDY0lcy9cmkEdzEwRVmKSiT0QIIons1Egd7NlbYNprhapqVnNFa1tdSTx+CU/4FTfYK8uLNGYgRh1fSiZHq6d2m6YQ5EEoElXStyNzr3gbxxLvMe0wFmJpKZOMmm0u1PmklKw2/KuAC0rkBa7pDUXFLw6powlB6i1nRN2H+NOlXQVFrViKJOrQBG+BXxj8hWrs5R10vk4Rk5Agpk3eFLn+0He4qbnKcQ15jVDvxz76EpAM1YsHtLgQws/Q3kb1OqoynMF5+uW9fkCnvobVQ+4Amvd9+2r23W6pAe42ufuZIqqKCPUCIvSFo+W1O0jq4Ggr3Kw+n+Oc5dThmHlKgV+UtCBEco6GEwgImkuR2bPijKlORknNl+bqghL720FJMbKOmnzSwCFUQqZ0Ol/QbPKg0aOWdYCGJ78sTOKAaWXt9OtGbdkATLjSONV026Q6/HsnxWQydIXaSrJd4KV9mu1wxViR9+dYlk3TrvZpS8jJw/vaA/p5tDmZh8BCI23J6KEvpzpITRICFnVhkzuYOmHDCZkzm9UpsZAD3ZYGulrafdFfXoOj9kWVnRuquF64ZOX1LaICMV1dX6ul3DRMxFGrmm8qv6eTvTtoK8VS7HqobsvFeYy47JxTp3eULvLvmQ49ZuUfuHfO22n44VHnCNGBNjKN1KNuJMOoRQYe8g04YtRQSzJvTtdi5EAK2cPOBQKwYBAdV026Gi8mkY6fbnHfaKoAnjGwGyRULHyDIP2hwfm3DWbrtNRPuExu/bbE8a4VcRlpABRCtQlN9Z6DUeN0MsM4uIQs6k4Yrjjis2Tvgg2Le98B1gLMtaXRlUMpZG5Zvg+RGgpQ0rnZ63a92PE7unQIN9UWrx7XDRD1VKQ2+1y1o5HSV7W88tNTbMwFiWj6vJ1vr1r67SzayhHmQpgHZEMR7IZ3qRZxAcm1Kl+qoxaj6OycXFxAC35y7EutTabUewgE9U0VN6qjb4IKlRte6xxGjHCap1M0rQ3Hk3W9rpcIhm+sK2oXNe3GalXXlNdLwxlmMbUwzUSlJ7JAPtRDQXKlY5x1SmEZkMgj2DAT3GrHMaM1NF9rXUrnOs+uubSPaaDSaWyU55BUTdFqXW6DtP+yjjkjMug45oh1IHYWZntBzt7ZsRHZv27BwyoAs1HrDbgs9Rl8ltuZulHiwFT0Gt1EqHysFktDYIBgoOlhAFyGwUN6Ig1MM9kN0O2kjd5jOh6RsG7L0iNXsOJGlALV6uluEiWxUl7mQ9Yc9eSNK3NQOLzMNzAaohwmWcriqhGr8YNMwrTIWIKPfHjfxgKuLErQfzxIjpaYNpkoV+xNOoOaLXOpYd/GIO44uBkIe4incG55B3S0HOAH7v9BhQ4ECMH4PAQHC1sLRIR9ct4d44dzoZO16mlU0Itng6XYBeuxArgn0OgQao78hZ5bkoQBFnlT36g290Vk0XdIe/kvudEaBtEZ+vaAvqkaMv+N/eVH2ou1ryNJ7BXG2zky2XR0PdxN7LVoBXL+ZcO2WexJVzyOhfdGTILdzL7W2gu0rxMa76aIpj1QisU46NKoFZjql8Qnu/sj3EWjeNnJVqILyrRQq1NlGnvA4YeOV4Zosd61oStJZ3el04jlCO9UqutHK57IR4Z5FR7ovcBtQDx7G/pf9DUBzKtdoGQYCYQfByD5Rb+0bb9epR1473jvFbabtZ25DZCy5+ad8opwGTbjN9NF8YQ4R0kMhtpbLoq6/Y98w162fIOcnBZ3opQt1Kv0OUAwk4nvZRPQHIFs55vw2C4EkRxpJs7rRjctwzLt2rhvlsiNuPSWXvjbxkJiVzoXdGjHgPRGHqn72xUOTbg/Xb1h0ejI1duNuMj6E6Y4dAAtWvOucIZLG9++KzFFPmBFhVJwxae6uknNJDQ9OgXPBcjFVkhfuu9Pg1G+63d1ukpwe60Phlo6py0K8kIiwKUS6yJDKxCDLHU7RZ2ScEP0/ZntzAljwg0UYERAdssq/1ijX15TU3g+U6/OY6xm7NaMdA01ftJo16iPZ0VuSaWhwGz2fyrEgtTckhnrBBHJZ7O153XY/8huwCD/JBf1gC+RIfOVMBbgUXkS/tG6Na+GMC0K8LUFRiQY7htSVZeXJFmnfTl4IHWlGoTgOe9mwi6RpNpXkEF/ttzpX0SbEmlRL3GshlSNiRaK8cxy1bLc6xJqMmnUlqchunZHQoc+NBs4J/bXuzmi1e4u6fa3b/9IqVGrnaFxpZMEnnBqG+KIoS7RgSz8s4EiMnnoNiYWQUp+R5TIk1tFr16qX1PmbIyoAdsCpLhKulPDQ0xCNwIo/iZWlt1/fRlnYaSq/ldmrgDKfgQcDv5lkZpzN5nYq5um5Tbguyh+Jd8lNwg9hBa1O1CpCr3z/wtowYpGkROWx4cKB375YybSPTBREtsq99fE5BVSCdxRu0XsiBRgP8l1+hfHmZi2k1cviqyqwTBmTI6OupE2RwVHuKXmj3WU/f9aS1HXMdVSmzUEbYdrb3qq0d+udC4CoJTOP0Qa4d2b5n06ZHP82+7n6slak26w8ksMdfi9M7b/LIhUgpRozuWC9vIe5VWM+gY09nDbF+ffan3yBzSvx7DvroHMPujw7UdChEs/tQiGaSgjxxT/sXs7heBL+LgDVkdbynORjv0YjTEaHmOFt7As2eO5VvOSVjJjwA34QR3120wPF+G/YseXlB+gXAu+SBvA3xbY1Q/LTiSe15MIfbCFeyncaoy7iIIbhUchqpVxFcgqyhgixKn+iUuOXcsNDzEBUeEc6sEVi29QfDMQgp2uDFXJgY9kUnPL8KkGm9vheaQTyEJ8f511vwy0ROIVgHLzdvBXaQx+wJQmO9I2jo6Oigxkhj5XWdrfb9jAj8msvGzp9536YTLq/DCrpvxsrrMC+y899Y/IxrLoe974jgWpNMdvB0toQb+8AbIMHeYDDUmITHUyB4lPV+FkVWhkl8IVxsbNfReeRg2z2LAQZwLw32pWt1AlDFc6/ZC5P6yRxZrdvoBah62Ua2qANlOUnhOYiKXlOxz+DJZx4uz1hUvWYsf9A4A9UgFwBwutS8VYxw2cw1Yd6alRpPO2c4zcFlmM44XQlrOKhf4VStnZmBYOn40LfHMbFOJFIa8GzPOp6SgwpU9KMzRzJOrexug1nf9FtTKKeQkqC7WU1dTa1+GqExUTFdwTPUTutwaLdJvaPcbazXuvTBX+ZeU/+Ui3udHeSGjqA6Wck4GoOGu+v2aip0CjdyaeIK8aOyCtZUoMVliivvhhQTugJBVfpoVZ+ELTZqY/qepjLGhD8nS/JZVzi2XUGD4c5Gxarnyu+dZb9nbnY2KmLbfDBfxfgjyFa4w9Y1Jr+1pqv8cMeoggI8TWJ62azVPVrcaPttmryMaRo26vI99Xg7Hf6Ane7qXS332mLCKHRnq3Hzea+9GuW7iu0s7c6cX03HH08vOzpJi4PGHA5bNiKnqDavp3bltvlZ8y4tz9Dow6VWFWtCeWtC+eYJ7T3AF/RN6G2Tyr9pUuVqTj2lej7bMmgr9RhDAvAr+l89puf01xzTYetMYr3zf8rTKFuyv64gm3E2ROYUpLLPWRlX8aVgp/c5rdOIzh9Gohmlf0rF7mW5e4r7e6efjrc9H4jbt/T+uISKcnSNgKhw8tD/PWzW1sEvHvbr7ONt2LXNiz9CxPz/KayF5KkVwwKzv/UY1nwhwWfyEwmtEKD/Cwp3R7bkfTRFujGUWs5fVzLPN0az/eQ0JMroc/sYNudRJKKwMVoUJWAAIIXtUyXSbxn9SiabfsTw+sAQoSMiv8u9NbJbw4ZbJ7yHrn/bVP6eY95fQvO/WevvpfHNGPhB8e7DjGDbKPk/bAu/sh10bKATkJ1Kedw3BvtsRPbtoZehtSuZ+QPFYSaiulcYJgp1gE6tgrbDMVOh9cr6Y8RfDz/5nRWRwBTOcBfgW5dqy9MdOyWIGAoccxZugtsSU5gb/OrOWL6Fjr8TAE2BKkVOFFApy60F43Xn+Bl1rOxvIfBFb3ny7Pk9Tl7j0t8LuRctydG2zvMDzzpzls1mwJXe01aIqBCQTt65u7J7zOvoBUB69W8A84xrshYM4eeSrBtUQtc0LbTF1ra7Z5+OOOd0aF2fOlu4a/a94Y2YH6vTMD7TRCV42wy3TppR2fYnzbRIoH+/PlmOWInctdjoe1OO1AeXYSXwqne8v8fDQoMDH99+QqEAMfrQCYS08EzfyUG5dOIIMeLapiFP+Ww6jXPQv/0EKD0GOmCU4+9IN7+b3EBoPrqmrwVhqVLe7yYIPJ5Dgw99fZ6JhoufJyrQrl1v0gHjeEnL8sFgfwNOf5nKsyP3g2l5Qs5V3D30nN1nUexKSm+HIFSjmnBtVPC3eOyuBkoyN5qFDZg6E5y+dCe/1Mbxxe9N3wbZ+DGQ1kdDSN9CqeNWsZxQq+B+3xChd/tsqu1Kvwh2V1luzsHaqG2P0UZruRcZLASP3GcDb6s2NYRvm4FboAwMSkiGkFpBMkmKju9m+dgWG72XbBfQIf5GAb4GA0M279M23rlZYG9jR6kMvXAD9/awoOgK6I0kC/2IOtgGTVFbcxryXj8uXG3zPm0HFt4q1h8EB6jm4IDzBPcSnRAdEXO8+qDyf+htOZnrbjhUq3eD8XsvhVgC7gNPMAZQSyt6ktb0GO/MZRnGGFjSWn9S29PG4T3p/wgItLcG4ber2y+t3l61/WJgX+1JW0hyytShmRYOWjKyk2rsrLHWZ+fC/Q/7T6XKFHBDAKvOw2iIlm8oQyX10U+5nGe3mNwXYu83nfaBsV9vTvsO/MoW9cmbjY3tYw/UyJpA356wW2hY68APJtHNYYiUKb9t/D0vDz2GFdDudy7og1rWd2y6oUE4xxdIDpnrzNWbJGcZAApTdwQL/q0E9Ms/rnyVRG5iIaEv9MKKvm1TAuksV/KTfRY1ooOvGIaRAHTjpbA/LsLswrKK6ko2ZTwv3RME1aT5eRliJSINBQwKwNfhi9J/O/pMi8HWuw0W4zcmP5IILf2/z7TnIh9JHy4h2euPeNY5Qecw06aATU5rBx7oy6ET7SBavdcHnCQfDZg2Pe/8Hz1vMbc=\"}")
for relative, encoded_content in encoded_files.items():
    destination = SOURCE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(zlib.decompress(base64.b64decode(encoded_content)))
print(f"Extracted {len(encoded_files)} versioned source/config files")


In [ ]:
PRESIGNED_CONFIG_ZLIB_B64 = ''
if PRESIGNED_CONFIG_ZLIB_B64:
    presigned_path = PROJECT_DIR / "s3_presigned_config.json"
    presigned_path.write_bytes(zlib.decompress(base64.b64decode(PRESIGNED_CONFIG_ZLIB_B64)))
    presigned = json.loads(presigned_path.read_text(encoding="utf-8"))
    os.environ["S3_PRESIGNED_CONFIG_PATH"] = str(presigned_path)
    os.environ["S3_BUCKET"] = presigned["bucket"]
    os.environ["S3_PREFIX"] = presigned["s3_prefix"]
    os.environ["RUN_ID"] = presigned["run_id"]
    os.environ["AWS_REGION"] = presigned["aws_region"]
    os.environ["AWS_DEFAULT_REGION"] = presigned["aws_region"]
    print("Loaded short-lived object-scoped S3 operations; no AWS key is embedded.")
else:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    aliases = {
        "AWS_ACCESS_KEY_ID": ("AWS_ACCESS_KEY_ID",),
        "AWS_SECRET_ACCESS_KEY": ("AWS_SECRET_ACCESS_KEY",),
        "AWS_REGION": ("AWS_REGION", "AWS_DEFAULT_REGION"),
        "AWS_DEFAULT_REGION": ("AWS_DEFAULT_REGION", "AWS_REGION"),
        "S3_BUCKET": ("S3_BUCKET",),
        "S3_PREFIX": ("S3_PREFIX",),
    }
    missing = []
    for environment_name, candidates in aliases.items():
        value = None
        for candidate in candidates:
            try:
                value = client.get_secret(candidate)
            except Exception:
                value = None
            if value:
                break
        if value:
            os.environ[environment_name] = value
        else:
            missing.append("/".join(candidates))
    if missing:
        raise RuntimeError("Missing S3 configuration: " + ", ".join(sorted(set(missing))))
os.environ["PYTHONHASHSEED"] = "2026"
print("S3 environment configured; credential values were not printed.")


In [ ]:
required = {
    "lightgbm": "lightgbm>=4.0,<5",
    "boto3": "boto3>=1.34,<2",
    "requests": "requests>=2.31,<3",
}
missing = []
for module, requirement in required.items():
    try:
        imported = __import__(module)
        if module == "lightgbm" and not str(imported.__version__).startswith("4."):
            missing.append(requirement)
    except ImportError:
        missing.append(requirement)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
import lightgbm as lgb
print(f"LightGBM={lgb.__version__}; device=CPU; GPU/TPU disabled")


In [ ]:
preferred = Path("/kaggle/input/cicddos2019-parquet-per-classes")
if preferred.exists():
    data_dir = preferred
else:
    parquet_files = sorted(Path("/kaggle/input").rglob("*.parquet"))
    if not parquet_files:
        raise FileNotFoundError("No Parquet files found in attached Kaggle inputs")
    data_dir = Path(os.path.commonpath([str(path.parent) for path in parquet_files]))
print(f"Preparing deterministic leakage-safe splits from {data_dir}")
data_command = [
    sys.executable, str(SOURCE_DIR / "data.py"),
    "--config", str(SOURCE_DIR / "config/data.smoke.json"),
    "--data-dir", str(data_dir),
    "--output-dir", str(PREPARED_DIR),
]
if os.environ.get("RUN_ID"):
    data_command.extend([
        "--s3-config", str(SOURCE_DIR / "config/train.smoke.json"),
        "--run-id", os.environ["RUN_ID"],
        "--maximum-hours", "12",
        "--stop-before-minutes", "30",
    ])
data_result = subprocess.run(data_command, cwd=SOURCE_DIR, check=False)
if data_result.returncode not in (0, 75):
    raise subprocess.CalledProcessError(data_result.returncode, data_command)
PREPROCESSING_PAUSED = data_result.returncode == 75
if PREPROCESSING_PAUSED:
    print("Preprocessing paused after a durable source-file checkpoint; training is deferred to the next session.")


In [ ]:
if PREPROCESSING_PAUSED:
    print("Skipping training in this session because preprocessing will resume first.")
else:
    train_command = [
    sys.executable, str(SOURCE_DIR / "train.py"),
    "--config", str(SOURCE_DIR / "config/train.smoke.json"),
    "--prepared-data-dir", str(PREPARED_DIR),
    "--output-dir", str(RUNS_DIR),
    "--upload-checkpoints-to-s3",
    ]

    if os.environ.get("RUN_ID"):
        train_command.extend(["--run-id", os.environ["RUN_ID"]])
    result = subprocess.run(train_command, cwd=SOURCE_DIR, check=False)
    if result.returncode not in (0, 75):
        raise subprocess.CalledProcessError(result.returncode, train_command)
    if result.returncode == 75:
        print("Session paused only after a verified checkpoint; the watchdog may launch the next session.")
    else:
        print("Training reached iteration 100 and final reporting completed or remains durably retryable.")


In [ ]:
active_path = RUNS_DIR / "active_run.json"
if active_path.exists():
    active = json.loads(active_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "run_id": active.get("run_id"),
        "status": active.get("status"),
        "current_iteration": active.get("current_iteration"),
    }, indent=2))
else:
    print("No active run pointer was created.")
